## Creating Training Subsets

Small notebook to cutdown on some of the SEC experiment training set sizes

In [1]:
import pandas as pd
import os
sample_seed = 42

In [5]:
train_data_path = '../sec/data/arc/train.parquet'
df_train = pd.read_parquet(train_data_path)
N = len(df_train)
print(f"Original train set size: {N}")
display(df_train.head())

Original train set size: 30000


,data_source,prompt,ability,reward_model,extra_info
0,arc-easy-train,[{'content': 'A conversation between User and ...,math,{'ground_truth': {'target': '0 0 0 0 0 0 1 1 1...,"{'difficulty': 1, 'index': 0, 'question': 'Fin..."
1,arc-easy-train,[{'content': 'A conversation between User and ...,math,{'ground_truth': {'target': '8 8 8 8 8 8 0 8 8...,"{'difficulty': 1, 'index': 1, 'question': 'Fin..."
2,arc-easy-train,[{'content': 'A conversation between User and ...,math,{'ground_truth': {'target': '0 0 0 0 1 1 1 1 2...,"{'difficulty': 1, 'index': 2, 'question': 'Fin..."
3,arc-easy-train,[{'content': 'A conversation between User and ...,math,{'ground_truth': {'target': '0 0 2 2 2 2 2 2 2...,"{'difficulty': 1, 'index': 3, 'question': 'Fin..."
4,arc-easy-train,[{'content': 'A conversation between User and ...,math,{'ground_truth': {'target': '8 8 8 8 8 8 8 8 2...,"{'difficulty': 1, 'index': 4, 'question': 'Fin..."


In [17]:
print(pd.unique(df_train.data_source))

# take 10% of original training set for now
subset_size = int(0.1 * N)
print(subset_size)

# even splits across subset size
split_size = subset_size // 3
print(split_size)

['arc-easy-train' 'arc-medium-train' 'arc-hard-train']
3000
1000


In [ ]:
# seeded random sample for each split
df_easy = df_train[df_train.data_source == 'arc-easy-train']
df_easy_subset = df_easy.sample(n=split_size, random_state = sample_seed)

df_med = df_train[df_train.data_source == 'arc-medium-train']
df_med_subset = df_med.sample(n=split_size, random_state = sample_seed)

df_hard = df_train[df_train.data_source == 'arc-hard-train']
df_hard_subset = df_hard.sample(n=split_size, random_state=sample_seed)

df_train_subset = pd.concat(
    [df_easy_subset,
     df_med_subset,
     df_hard_subset
    ]
)

len(df_train_subset)

3000

In [23]:
# save to parquet
os.makedirs('../data/arc', exist_ok=True)
df_train_subset.to_parquet("../data/arc/train_subset.parquet")

In [ ]:
# NOTE: this is actually pretty small so keep as is...
test_data_path = '../sec/data/arc/test.parquet'
df_test = pd.read_parquet(test_data_path)
N = len(test_data_path)
print(f"Original test set size: {N}")

Original test set size: 28


## Creating GSM subset

In [3]:
# finding intersect between original verl preprocessed parquet and our subset

train_data_path = '/home/vince/data/gsm8k/train.parquet'
df_train = pd.read_parquet(train_data_path)
N = len(df_train)
print(f"Original train set size: {N}")
display(df_train.head())

Original train set size: 7473


,data_source,prompt,ability,reward_model,extra_info
0,openai/gsm8k,[{'content': 'Natalia sold clips to 48 of her ...,math,"{'ground_truth': '72', 'style': 'rule'}",{'answer': 'Natalia sold 48/2 = <<48/2=24>>24 ...
1,openai/gsm8k,[{'content': 'Weng earns $12 an hour for babys...,math,"{'ground_truth': '10', 'style': 'rule'}",{'answer': 'Weng earns 12/60 = $<<12/60=0.2>>0...
2,openai/gsm8k,[{'content': 'Betty is saving money for a new ...,math,"{'ground_truth': '5', 'style': 'rule'}","{'answer': 'In the beginning, Betty has only 1..."
3,openai/gsm8k,[{'content': 'Julie is reading a 120-page book...,math,"{'ground_truth': '42', 'style': 'rule'}",{'answer': 'Maila read 12 x 2 = <<12*2=24>>24 ...
4,openai/gsm8k,[{'content': 'James writes a 3-page letter to ...,math,"{'ground_truth': '624', 'style': 'rule'}",{'answer': 'He writes each friend 3*2=<<3*2=6>...


In [14]:
df_train_content = [x[0]['content'] for x in df_train.prompt]

In [4]:
train_data_subset_path = '/home/vince/data/gsm_1k.parquet'
df_train_subset = pd.read_parquet(train_data_subset_path)
N = len(df_train_subset)
print(f"train subset size: {N}")
display(df_train_subset.head())


train subset size: 1000


,question,answer
0,In Professor Plum's biology class there are 40...,"We start with the initial numbers of students,..."
1,Diane bought twenty more apples than Cecile. I...,Diane bought 15 + 20 = <<15+20=35>>35 apples.\...
2,Ann can skate 6 miles an hour. Her friend Glen...,First find how far Glenda goes in 3 hours by m...
3,"Running for 2 hours, Jonah burnt 30 calories e...","When Jonah ran for 2 hours, burning 30 calorie..."
4,The city of Richmond has 1000 more people than...,Victoria has 3000-1000=<<3000-1000=2000>>2000 ...


In [24]:
sample = df_train_subset.question[0]
sample

"In Professor Plum's biology class there are 40 students. Of those students, 80 percent have puppies. Of those who have puppies, 25% also have parrots. How many students have both puppies and parrots?"

In [29]:
df_train_content[1297].strip('\\')

'In Professor Plum\'s biology class there are 40 students. Of those students, 80 percent have puppies. Of those who have puppies, 25% also have parrots. How many students have both puppies and parrots? Let\'s think step by step and output the final answer after "####".'

In [ ]:
# from fuzzy_match import match
from thefuzz import process

matched_idxs = []
for i, sample in enumerate(df_train_subset.question):
    # print(i)
    matched_str = process.extractOne(sample, df_train_content)[0]
    # print(matched_str)
    idx = df_train_content.index(matched_str)
    matched_idxs.append(idx)
    # break



In [56]:
df_out = df_train.iloc[matched_idxs]
df_out

,data_source,prompt,ability,reward_model,extra_info
1297,openai/gsm8k,[{'content': 'In Professor Plum's biology clas...,math,"{'ground_truth': '8', 'style': 'rule'}",{'answer': 'We start with the initial numbers ...
576,openai/gsm8k,[{'content': 'Diane bought twenty more apples ...,math,"{'ground_truth': '50', 'style': 'rule'}",{'answer': 'Diane bought 15 + 20 = <<15+20=35>...
5462,openai/gsm8k,[{'content': 'Ann can skate 6 miles an hour. H...,math,"{'ground_truth': '42', 'style': 'rule'}",{'answer': 'First find how far Glenda goes in ...
4336,openai/gsm8k,"[{'content': 'Running for 2 hours, Jonah burnt...",math,"{'ground_truth': '90', 'style': 'rule'}","{'answer': 'When Jonah ran for 2 hours, burnin..."
7105,openai/gsm8k,[{'content': 'The city of Richmond has 1000 mo...,math,"{'ground_truth': '500', 'style': 'rule'}",{'answer': 'Victoria has 3000-1000=<<3000-1000...
...,...,...,...,...,...
4511,openai/gsm8k,[{'content': 'Gerald brings chocolate bars to ...,math,"{'ground_truth': '7', 'style': 'rule'}",{'answer': 'The teacher brings 14 bars to scho...
6930,openai/gsm8k,[{'content': 'Josh has 9 dollars. He spent $1....,math,"{'ground_truth': '6', 'style': 'rule'}",{'answer': 'Josh spent 1.75+1.25=<<1.75+1.25=3...
1057,openai/gsm8k,[{'content': 'Twenty gallons of tea were poure...,math,"{'ground_truth': '7', 'style': 'rule'}",{'answer': '20 gallons = 160 pints 160/80 = <<...
2221,openai/gsm8k,[{'content': 'Diana earned $150 in July. She e...,math,"{'ground_truth': '1500', 'style': 'rule'}",{'answer': 'The amount of money Diana earned i...


In [58]:
df_out.to_parquet(os.path.join('../data/gsm', 'train_subset.parquet'))

In [ ]:
# verify
df_train = pd.read_parquet('../data/gsm/train_subset.parquet')
df_train

In [ ]:
df_test = pd.read_parquet('../data/gsm/test.parquet')
df_test